# 🔍 ColdStart Killer — Buyer Search Demo

Notebook này demo full buyer search pipeline end-to-end từ raw user query đến top sản phẩm có explainability.

Pipeline demo:

1. User nhập query tự nhiên bằng tiếng Việt hoặc tiếng Anh.
2. Query Processor detect ngôn ngữ, translate sang English nếu cần, extract hard filters, tạo HyPE query và BM25 query.
3. Query được embed bằng `BAAI/bge-m3` thông qua `src.embeddings.embed_one()` — cùng embedding path với teammate indexing.
4. MongoDB chạy hybrid retrieval: `$vectorSearch` cho HyPE intent + `$search` BM25 cho facts/keywords.
5. `$unionWith` fallback + RRF fusion rank kết quả, `$lookup` sang `items`, scoring và output explainable JSON.
6. Notebook hiển thị kết quả, cold-start highlight và debug scoring.

Ownership:

- Teammate machine: dataset, LLM generation, embedding model, indexing, seller-side insert.
- Buyer search machine: query processing wrapper, MongoDB search pipeline, aggregation, ranking, output formatting.

Lưu ý: Notebook này cần chạy trên máy có Ollama/Qwen3 và BAAI/bge-m3 nếu query tiếng Việt cần translation và embedding runtime.

In [ ]:
# Setup: imports và kết nối MongoDB
# Cell này chuẩn bị sys.path, import các module project, kiểm tra MongoDB,
# và load embedding model BAAI/bge-m3 từ src.embeddings.

import sys
import json
from pathlib import Path
import pymongo

# Đảm bảo notebook import được src/* dù chạy từ repo root hay thư mục notebooks/.
ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.embeddings import load_embedding_model
from src.mongodb import get_items_collection, get_retrieval_units_collection, ping_mongodb
from src.query_processor import process_query
from src.search_pipeline import run_search

# Kết nối MongoDB và lấy collections chính.
ping = ping_mongodb()
print("MongoDB:", "CONNECTED" if ping.get("ok") else "FAILED")
if not ping.get("ok"):
    raise RuntimeError(ping.get("error", "MongoDB connection failed"))

items = get_items_collection()
retrieval_units = get_retrieval_units_collection()

# In collection counts để xác nhận dữ liệu indexing đã sẵn sàng.
counts = {
    "items": items.count_documents({}),
    "retrieval_units": retrieval_units.count_documents({}),
    "hype_question_with_embedding": retrieval_units.count_documents({"unit_type": "hype_question", "embedding": {"$exists": True}}),
    "proposition_with_text_search": retrieval_units.count_documents({"unit_type": "proposition", "text_search": {"$exists": True}}),
}
print("\nCollection counts")
for name, value in counts.items():
    print(f"- {name}: {value}")

# Load embedding model. Bước này cần teammate machine có BAAI/bge-m3 dependencies/model cache.
print("\nLoading embedding model BAAI/bge-m3 via src.embeddings...")
embedding_model = load_embedding_model()
print("Embedding model loaded:", type(embedding_model).__name__)


# User Query Input

Đây là điểm bắt đầu của demo. Thay đổi `QUERY` để test các loại query khác nhau.

In [ ]:
# ← THAY ĐỔI QUERY Ở ĐÂY ĐỂ TEST
# Example queries:
# "moisturizing cream for dry skin"
# "phone case samsung galaxy s22 under 300k"
# "váy đầm dự tiệc đẹp"
# "wireless charger iphone 14"
# "tai nghe chống ồn dưới 500k"

QUERY = "tai nghe chống ồn dưới 500k"
TOP_K = 10

print("QUERY:", QUERY)
print("TOP_K:", TOP_K)


## 🧠 Bước 1: Query Processing

Ở bước này, raw query được chuyển thành fixture search-ready trong memory: detect ngôn ngữ, translate nếu cần, extract filters, tạo HyPE semantic query, tạo BM25 keyword query, và embed query bằng BAAI/bge-m3.

In [ ]:
# Bước 1: Xử lý query — detect ngôn ngữ, translate, extract filters, embed
# process_query() trả về dict có thể đưa trực tiếp vào run_search().

fixture = process_query(QUERY)

# In từng kết quả trung gian để demo rõ pipeline đang làm gì.
print("Language detected:", fixture["language_detected"])
print("English translation:", fixture["english_query"])
print("Hard filters extracted:", json.dumps(fixture["hard_filters"], ensure_ascii=False))
print("HyPE query built:", fixture["hype_search_query_en"])
print("BM25 query built:", fixture["bm25_search_query_en"])

embedding = fixture["query_embedding"]
print("Embedding dimension:", len(embedding))
print("Embedding first 3 values:", embedding[:3])


## 🔎 Bước 2: Hybrid Search (Vector + BM25)

MongoDB chạy hai kênh retrieval: `$vectorSearch` trên HyPE embeddings để bắt semantic buyer intent, và `$search` BM25 để bắt keyword/fact matches. Demo dùng `unionWith` fallback với RRF = `weight / (60 + rank)` để chạy ổn định trên Atlas tier không hỗ trợ native `$rankFusion`.

In [ ]:
# Bước 2: Chạy hybrid search — $vectorSearch + $search BM25 + RRF fusion
# mode="unionWith" dùng fallback ổn định, không phụ thuộc native $rankFusion preview.

SEARCH_MODE = "unionWith"
results = run_search(fixture, top_k=TOP_K, mode=SEARCH_MODE)

print("Search mode used:", SEARCH_MODE)
print("Number of results returned:", len(results))


## 📊 Bước 3: Kết quả tìm kiếm

Bảng dưới đây hiển thị top results kèm score, rank từ từng channel, matched intent/fact, và cold-start signal.

In [ ]:
# Bước 3: Hiển thị kết quả với explainable output
# Không dùng pandas để notebook nhẹ và dễ chạy trên mọi máy.

def short_text(value, max_len=60):
    text = "" if value is None else str(value).replace("\n", " ")
    return text if len(text) <= max_len else text[: max_len - 3] + "..."

def format_vnd(value):
    if value in (None, ""):
        return ""
    try:
        return f"{int(value):,} VND"
    except (TypeError, ValueError):
        return str(value)

def channels_for(result):
    channels = (result.get("debug") or {}).get("matched_channels") or []
    if isinstance(channels, list):
        return channels
    return [channels]

print("Rank | Title | Score | Price | Brand | Category | Vector Rank | BM25 Rank | Channels | Matched Intent | Matched Fact | Cold Start")
print("---: | --- | ---: | --- | --- | --- | ---: | ---: | --- | --- | --- | ---")
for rank, result in enumerate(results, start=1):
    debug = result.get("debug") or {}
    print(
        f"{rank} | "
        f"{short_text(result.get('title'))} | "
        f"{result.get('score')} | "
        f"{format_vnd(debug.get('price_vnd'))} | "
        f"{debug.get('brand') or ''} | "
        f"{debug.get('category_id') or ''} | "
        f"{result.get('rank_vector')} | "
        f"{result.get('rank_bm25')} | "
        f"{', '.join(str(channel) for channel in channels_for(result))} | "
        f"{short_text(result.get('matched_intent'))} | "
        f"{short_text(result.get('matched_fact'))} | "
        f"{'yes' if result.get('cold_start_note') else 'no'}"
    )


## ❄️ Cold Start Analysis

Cold-start items thường chưa có interaction history, nên collaborative filtering truyền thống khó rank chúng. ColdStart Killer dùng HyPE intent + proposition facts để surface item dựa trên nội dung và intent match ngay sau khi indexing.

In [ ]:
# Cold Start Analysis — items nào được tìm thấy nhờ HyPE/proposition
# Lọc các result có cold_start_note để giải thích vì sao item cold-start xuất hiện.

cold_results = [result for result in results if result.get("cold_start_note")]
print("Cold-start items in results:", len(cold_results))

if not cold_results:
    print("No cold-start items found in this result set.")
else:
    print("Rank | Item ID | Why it appeared | Matched Intent | Matched Fact")
    print("---: | --- | --- | --- | ---")
    for rank, result in enumerate(cold_results, start=1):
        reason = "HyPE/vector matched buyer intent; BM25/proposition matched factual or keyword evidence."
        print(
            f"{rank} | "
            f"{result.get('item_id')} | "
            f"{reason} | "
            f"{short_text(result.get('matched_intent'), 80)} | "
            f"{short_text(result.get('matched_fact'), 80)}"
        )


## 🔬 Explainability — Tại sao item này được rank #1?

Debug view giúp xem rank từng channel, fusion score, bonus và matched channels của item top 1.

In [ ]:
# Debug view — xem chi tiết scoring cho item top 1
# In full debug dict và các score/rank chính để teammate dễ explain trong video.

if not results:
    print("No results to debug.")
else:
    top_result = results[0]
    debug = top_result.get("debug") or {}
    print("Top item:", top_result.get("item_id"), "-", top_result.get("title"))
    print("raw_vector_score:", debug.get("raw_vector_score"))
    print("raw_bm25_score:", debug.get("raw_bm25_score"))
    print("rank_vector:", top_result.get("rank_vector"))
    print("rank_bm25:", top_result.get("rank_bm25"))
    print("fusion_score:", top_result.get("fusion_score"))
    print("multi_channel_bonus:", debug.get("multi_channel_bonus"))
    print("cold_start_boost:", debug.get("cold_start_boost"))
    print("matched_channels:", debug.get("matched_channels"))
    print("\nFull debug dict")
    print(json.dumps(debug, indent=2, ensure_ascii=False, default=str))


## ✅ Demo Summary

In [ ]:
# Tổng kết demo thành bảng ngắn để biết pipeline đã sẵn sàng record video chưa.

query_processed = bool(fixture.get("query_embedding")) and len(fixture.get("query_embedding", [])) == 1024
filters = fixture.get("hard_filters") or {}
filters_applied = bool(filters)
channel_set = set()
for result in results:
    channel_set.update(channels_for(result))
hybrid_label = "vector + bm25" if {"vector", "bm25"}.issubset(channel_set) else "vector only"
cold_count = len([result for result in results if result.get("cold_start_note")])
pipeline_ready = query_processed and len(results) > 0

print("Metric | Value")
print("--- | ---")
print(f"Query processed | {'YES' if query_processed else 'NO'}")
print(f"Language detected | {fixture.get('language_detected')}")
print(f"Hard filters applied | {'YES' if filters_applied else 'NO'} - {json.dumps(filters, ensure_ascii=False)}")
print(f"Results returned | {len(results)} items")
print(f"Hybrid channels | {hybrid_label}")
print(f"Cold start items in top 10 | {cold_count}")
print(f"Pipeline status | {'READY' if pipeline_ready else 'NOT READY'}")
